# Authors Data Generator

Generates JS data modules for the Authors section visualizations.

**Output files** (in `site/data/authors/`):
- `authorMetricsData.js` → authorMetrics.js
- `authorsPerPaperData.js` → authorsPerPaper.js
- `collaborationNetworkData.js` → collaborationNetwork.js
- `uniqueAuthorsData.js` → uniqueAuthorsTimeline.js

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Paths
DATA = Path("../data/processed/outputs/openalex_notebook_outputs/tables")
OUT = Path("../site/data/authors")
OUT.mkdir(parents=True, exist_ok=True)

print(f"Source: {DATA}")
print(f"Output: {OUT}")

Source: ../data/processed/outputs/openalex_notebook_outputs/tables
Output: ../site/data/authors


---
## 1. Author Metrics (Bubble Chart)

In [2]:
# Load author stats
df = pd.read_csv(DATA / "sec2d_author_stats.csv")

# Categorize authors
def categorize(r):
    if r["papers"] >= 30: return "prolific"
    if r["citations"] >= 3000: return "highly-cited"
    if r["papers"] >= 10: return "steady"
    return "emerging"

df["category"] = df.apply(categorize, axis=1)

# Top 100 authors by papers + citations
df["score"] = df["papers"] + df["citations"] / 100
top = df.nlargest(100, "score")

# Build data
data = top[["author_name", "papers", "citations", "awards", "category"]].copy()
data.columns = ["name", "papers", "citations", "awards", "category"]
data["awards"] = data["awards"].fillna(0).astype(int)
records = data.to_dict(orient="records")

# Stats with categories descriptions (required by JS)
stats = {
    "totalAuthors": len(df),
    "maxPapers": int(df["papers"].max()),
    "maxCitations": int(df["citations"].max()),
    "maxAwards": int(df["awards"].max()),
    "avgPapers": round(df["papers"].mean(), 1),
    "avgCitations": round(df["citations"].mean(), 1),
    "categories": {
        "prolific": "30+ papers",
        "highly-cited": "3000+ citations",
        "steady": "10+ papers",
        "emerging": "New authors"
    }
}

# JS output
js = (
    "/** AUTO-GENERATED */\n"
    f"export const authorMetricsData = {json.dumps(records)};\n\n"
    f"export const authorMetricsStats = {json.dumps(stats)};\n"
)
(OUT / "authorMetricsData.js").write_text(js)
print(f"✓ authorMetricsData.js | {len(records)} authors")

✓ authorMetricsData.js | 100 authors


---
## 2. Authors per Paper (Line Chart)

In [3]:
# Load authors per paper by year
df = pd.read_csv(DATA / "sec2a_authors_per_paper_by_year.csv")

# Build data with error bands
records = []
for _, r in df.iterrows():
    avg = r["avg_authors"]
    var = r.get("std_authors", avg * 0.3)  # fallback variance
    records.append({
        "year": int(r["Year"]),
        "avg": round(avg, 2),
        "variance": round(var, 2),
        "min": round(max(1, avg - var), 2),
        "max": round(avg + var, 2)
    })

# Stats
stats = {
    "overallAvg": round(np.mean([r["avg"] for r in records]), 2),
    "minYear": min(r["year"] for r in records),
    "maxYear": max(r["year"] for r in records),
    "trend": "increasing" if records[-1]["avg"] > records[0]["avg"] else "stable"
}

# JS output
js = (
    "/** AUTO-GENERATED */\n"
    f"export const authorsPerPaperData = {json.dumps(records)};\n\n"
    f"export const authorsPerPaperStats = {json.dumps(stats)};\n"
)
(OUT / "authorsPerPaperData.js").write_text(js)
print(f"✓ authorsPerPaperData.js | {len(records)} years")

✓ authorsPerPaperData.js | 35 years


---
## 3. Unique Authors Timeline (Area Chart)

In [4]:
# Load cumulative unique authors
df = pd.read_csv(DATA / "sec2b_unique_authors_cumulative.csv")

# Build data
records = []
prev = 0
for _, r in df.iterrows():
    cum = int(r["cumulative_unique_authors"])
    records.append({
        "year": int(r["Year"]),
        "cumulative": cum,
        "newAuthors": cum - prev
    })
    prev = cum

# Stats
stats = {
    "total": records[-1]["cumulative"],
    "peakNewYear": max(records, key=lambda x: x["newAuthors"])["year"],
    "peakNewCount": max(r["newAuthors"] for r in records),
    "avgNewPerYear": round(np.mean([r["newAuthors"] for r in records]), 1)
}

# JS output
js = (
    "/** AUTO-GENERATED */\n"
    f"export const uniqueAuthorsData = {json.dumps(records)};\n\n"
    f"export const uniqueAuthorsStats = {json.dumps(stats)};\n"
)
(OUT / "uniqueAuthorsData.js").write_text(js)
print(f"✓ uniqueAuthorsData.js | total: {stats['total']:,} authors")

✓ uniqueAuthorsData.js | total: 7,219 authors


---
## 4. Collaboration Network (Force Graph)

In [5]:
import pandas as pd
import numpy as np
import json
import ast
from collections import defaultdict

# Load data
edges = pd.read_csv(DATA / "coauthor_edges.csv")
authors_stats = pd.read_csv(DATA / "sec2d_author_stats.csv")
authors = pd.read_csv(DATA / "authors.csv")
authorships = pd.read_csv(DATA / "authorships.csv")
institutions = pd.read_csv(DATA / "institutions.csv")

TOP_N = 50

# ----------------------------
# 1) Top authors by collaborations (degree in edge list)
# ----------------------------
collab_counts = pd.concat([edges["author_a"], edges["author_b"]]).value_counts()
top_ids = set(collab_counts.head(TOP_N).index)

# Keep only edges among top authors
edges_top = edges[(edges["author_a"].isin(top_ids)) & (edges["author_b"].isin(top_ids))].copy()

# ----------------------------
# 2) Compute "primary institution" per author (most frequent institution in authorships)
# ----------------------------
def parse_inst_list(s):
    if not isinstance(s, str) or not s:
        return []
    try:
        return ast.literal_eval(s)
    except Exception:
        return []

authorships = authorships.copy()
authorships["inst_list"] = authorships["institutions"].apply(parse_inst_list)

expl = authorships.explode("inst_list")
expl = expl[expl["inst_list"].notna() & (expl["inst_list"] != "")].copy()

inst_counts = expl.groupby(["author_id", "inst_list"]).size().reset_index(name="n")
inst_counts = inst_counts.sort_values(["author_id", "n"], ascending=[True, False])
top_inst = inst_counts.drop_duplicates("author_id").merge(
    institutions,
    left_on="inst_list",
    right_on="institution_id",
    how="left"
)

# Lookup maps
author_stats_map = authors_stats.set_index("author_id").to_dict(orient="index")
author_name_map = authors.set_index("author_id")["author_name"].to_dict()
top_inst_map = top_inst.set_index("author_id").to_dict(orient="index")

# ----------------------------
# 3) For each author, compute top collaborator (max edge weight among TOP_N subgraph)
# ----------------------------
adj = defaultdict(list)
for _, r in edges_top.iterrows():
    a, b, w = r["author_a"], r["author_b"], int(r["weight"])
    adj[a].append((b, w))
    adj[b].append((a, w))

top_collab = {}
for aid in top_ids:
    lst = adj.get(aid, [])
    if not lst:
        top_collab[aid] = (None, 0)
    else:
        best_id, best_w = max(lst, key=lambda x: x[1])
        top_collab[aid] = (best_id, int(best_w))

# ----------------------------
# 4) Build nodes with REAL groups = world_region
# ----------------------------
nodes = []
group_labels = []

for aid in top_ids:
    info = author_stats_map.get(aid, {})
    inst = top_inst_map.get(aid, {})

    name = info.get("author_name") or author_name_map.get(aid) or aid
    papers = int(info.get("papers", 0))
    citations = int(info.get("citations", 0))
    awards = int(info.get("awards", 0))

    collaborations = int(collab_counts.get(aid, 0))

    institution_name = inst.get("institution_name")
    country = inst.get("country_name") or "Unknown"
    region = inst.get("world_region") or "Unknown"   # <-- GROUP BASE

    best_id, best_w = top_collab.get(aid, (None, 0))
    best_name = None
    if best_id:
        best_name = (
            author_stats_map.get(best_id, {}).get("author_name")
            or author_name_map.get(best_id)
            or best_id
        )

    group_labels.append(region)

    nodes.append({
        "id": name,                     # what your current JS uses as node id/label
        "author_id": aid,
        "papers": papers,
        "citations": citations,
        "awards": awards,
        "collaborations": collaborations,
        "institution": institution_name,
        "country": country,
        "region": region,
        "top_collaborator": best_name,
        "top_collaborator_weight": best_w,
        "group_label": region           # keep explicit label
    })

# Map group label -> numeric group id (stable and meaningful)
unique_groups = sorted(set(group_labels))
group_id_map = {lab: i + 1 for i, lab in enumerate(unique_groups)}
for n in nodes:
    n["group"] = group_id_map[n["group_label"]]

# ----------------------------
# 5) Build links (still using author names as your JS expects)
# ----------------------------
name_map = {n["author_id"]: n["id"] for n in nodes}
links = []
for _, r in edges_top.iterrows():
    src = name_map.get(r["author_a"])
    tgt = name_map.get(r["author_b"])
    if src and tgt:
        links.append({"source": src, "target": tgt, "value": int(r["weight"])})

network = {"nodes": nodes, "links": links}

stats = {
    "totalNodes": len(nodes),
    "totalLinks": len(links),
    "avgCollaborations": round(float(np.mean([n["collaborations"] for n in nodes])), 1),
    "maxCollaborations": int(max(n["collaborations"] for n in nodes)),
    # group id -> group label (required by JS legend)
    "groups": {str(group_id_map[k]): k for k in group_id_map}
}

# ----------------------------
# 6) Write JS output
# ----------------------------
js = (
    "/** AUTO-GENERATED */\n"
    f"export const collaborationNetworkData = {json.dumps(network, ensure_ascii=False)};\n\n"
    f"export const networkStats = {json.dumps(stats, ensure_ascii=False)};\n"
)
(OUT / "collaborationNetworkData.js").write_text(js, encoding="utf-8")
print(f"✓ collaborationNetworkData.js | {len(nodes)} nodes, {len(links)} links | groups={stats['groups']}")


✓ collaborationNetworkData.js | 50 nodes, 133 links | groups={'1': 'Americas', '2': 'Asia', '3': 'Europe', '4': 'Oceania'}


---
## Summary

Generated files:
```
site/data/authors/
├── authorMetricsData.js
├── authorsPerPaperData.js
├── collaborationNetworkData.js
└── uniqueAuthorsData.js
```